# Kármán vortex street with the common MORFE API

Two-dimensional incompressible flow past a cylinder loses stability at $Re_c \approx 49$ through a Hopf bifurcation: the Kármán vortex street. This example reduces the 57 860-DOF Navier–Stokes model to a **single complex ODE**, the Stuart–Landau equation

$$\dot z_1 = \lambda z_1 + c_{101}\,z_1\eta' + c_{210}\,z_1|z_1|^2 + \dots$$

The Reynolds number rides along as a parametric coordinate $\eta' = 1/Re - 1/Re_0$, so one run at the expansion point describes the whole bifurcation neighbourhood. Steps 1 to 4 build the reduction; steps 5 to 7 read the limit-cycle branch, the shedding frequency and the peak lift straight off it. The committed output uses order 3. To reproduce the order-9 reference, change `order = 3` to `order = 9` and rerun the notebook.

Before the first run, run `julia setup.jl` in the bash from this repository. For more information look at the README.md of the repository.

`NSE` is only a short name for the fluid backend used to describe the flow case; `build_model`, `parametrise` and `normal_form_branch` belong to the common API.

In [ ]:
using MORFE, MORFEFerrite
const NSE = MORFEFerrite.FluidNavierStokes # for incompressible Navier-Stokes

## 1. Describe the fluid case

Choose the polynomial expansion order directly: `3` is the quick demonstration and `9` is the reference calculation.

`fluid_model` reads the mesh and assembles everything the flow problem needs: the P2/P1 Taylor-Hood spaces and their boundary conditions, the Newton solve for the steady base flow, the operators of the system linearised about it, and the viscous pieces that carry the Reynolds dependence. `Re = 49.03` is the expansion point, just above the critical Reynolds number. The mesh is a Turek–Schäfer channel with a ⌀0.1 m cylinder; its diameter is the reference length in $\nu = D/Re$.

In [ ]:
order = 3  # Change to 9 for the reference calculation.
case = NSE.fluid_model(joinpath(@__DIR__, "cylinder_flow.msh"); Re = 49.03)

## 2. Select the Hopf pair

Which modes span the manifold is a modelling decision, so it is made here in the driver: the backend has no policy about it. The two master coordinates are the Hopf pair; every other computed mode is an off-manifold target used only for diagnostics.

The eigenproblem $-B_0\,y = \lambda B_1 y$ is solved by shift-invert ARPACK. Because the shift $\sigma$ is complex, ARPACK returns only the modes near $\sigma$, and a strongly oscillatory mode's conjugate sits near $\bar\sigma$ and is never computed. `solve_hopf_eigenproblem` appends those missing halves itself, which is exact for real $B_0$ and $B_1$, so `conjugate_index` names the true partner of `hopf_index`. `case.B` is the pair $(B_0, B_1)$, the assembled model's linear operators in the order MORFE's model wants them. The descriptor system is $B_1\dot x + B_0 x = F$, so its eigenproblem carries a sign that belongs to the equation rather than to the caller; passing the tuple whole keeps it there.

In [ ]:
spectrum = NSE.solve_hopf_eigenproblem(case.B;
    nev = 40, sigma_re = 3.0, sigma_im = 8.0)

master = [spectrum.hopf_index, spectrum.conjugate_index]
outer = setdiff(eachindex(spectrum.eigenvalues), master)
spectrum.eigenvalues[master] # print the Hopf pair

## 3. Build the MORFE model

`build_model` converts the fluid case and its spectrum into MORFE's physics-independent model and spectral data. It computes the left eigenvectors of the master modes only, each costing its own adjoint factorisation, and conjugates the partner of a pair rather than solving for it, so the conjugate symmetry the reduction relies on holds by construction.

`scale = 1e-2` is a mode gauge: a uniform factor on both eigenvector sides, for conditioning. It changes what `W` and `R` mean, so coefficients from two gauges are not comparable, which is why it is written at the call site rather than hidden inside a default. The returned `meta` contains auxiliary backend information, including the conjugate permutation displayed below.

In [ ]:
(; model, spectral, meta) = build_model(case, spectrum;
    master = master, outer = outer, expansion_order = order, scale = 1e-2)
meta.conjugate_permutation # print the conjugate permutation

## 4. Parametrise the invariant manifold

`parametrise` computes the polynomial manifold map `W` and its reduced dynamics `R` up to the chosen order. The resonance configuration keeps near-resonant monomials in complex normal form; `tol_relative = 0.1` judges each target on its own frequency scale, flagging a monomial whose detuning is under 10 % of that target's $|\lambda|$. `outer_targets = true` also measures the monomials against the off-manifold modes, a diagnostic only, since the solve reads the master block alone.

Evaluating `R` at the end of the cell displays the reduced system: row 1 is the Stuart–Landau equation, row 2 its conjugate, and row 3 is $\dot\eta' = 0$, the frozen Reynolds coordinate. The Hopf bifurcation is supercritical when $\mathrm{Re}\,c_{210} < 0$.

In [ ]:
W, R = parametrise(model, spectral, order;
    resonance = ResonanceConfig(style = :complex_normal_form, tol_relative = 0.1,
        outer_targets = true))
R # print reduced dynamics

## 5. Read the bifurcation diagram off `R`

`R` already contains the answer. With a single conjugate pair in complex normal form every surviving monomial has $a - b = 1$, so substituting $z_1 = \rho e^{i\theta}$ gives every term the same factor $e^{i\theta}$ and $R_1$ loses its phase dependence. Matching $\dot z_1 = (\dot\rho + i\rho\dot\theta)e^{i\theta}$ against it separates amplitude from phase:

$$\dot\rho = \mathrm{Re}\,R_1(\rho, \rho, \eta') \qquad\qquad \Omega = \frac{\mathrm{Im}\,R_1(\rho, \rho, \eta')}{\rho}$$

A limit cycle is a nonzero root of $\mathrm{Re}\,R_1(\rho, \rho, \eta') = 0$, and $\Omega$ read there is its frequency. Each one is a whole periodic orbit, the circle $z_1 = \rho e^{i\Omega t}$, so the branch is really a paraboloid of cycles growing out of the base flow at $\mathrm{Re}_c$; the amplitude plots further down are that surface seen edge on.

`normal_form_branch` does all of it and returns plain vectors. At fixed $\rho$ the equation is a *polynomial* in $\eta'$, so the branch is solved for the parameter rather than continued in it: one companion-matrix root-solve per amplitude, with no step size, no initial guess and no fold to chase, since a fold in $\rho$ against Re is just a monotone $\eta'(\rho)$.

Two keywords carry the modelling decisions. `parameter_range` keeps the sweep physical, because the $\eta'$-polynomial also has roots far outside the expansion neighbourhood: at order 3 a second one at $\eta' \approx 0.071$, which is Re $\approx 11$. And `sheet = :primary` follows the branch out of the Hopf point rather than returning every root, which from order 9 includes a second sheet at large amplitude.

In [ ]:
Re₀ = 49.03
to_Re(η) = 1 / (η + 1 / Re₀)
to_η(Re) = 1 / Re - 1 / Re₀

sweep = (parameter = 1, sheet = :primary, amplitudes = range(0, 4; length = 500),
    parameter_range = (to_η(70.0), to_η(48.0)))

branch = normal_form_branch(R; sweep...)
Re_c = to_Re(branch.parameter[1]) # the ρ = 0 end of the branch is the Hopf point

## 6. A physical observable: the lift

$\rho$ is measured in the normal-form coordinate, whose scale is the mode gauge set in step 3,
so it is not a quantity anyone measures. Physical observables are projections of `W`, and once
the manifold is known they cost no full-order work: project the functional through `W` once,
then evaluate a small polynomial.

`lift_functional` returns the pressure-traction weight vector on the cylinder, and
`lift_polynomial` pushes it through `W` to give $L(z_1, \bar z_1, \eta')$, one complex
coefficient per monomial. On the limit cycle the orbit is the circle $z_1 = \rho e^{i\theta}$,
so the peak lift is a maximum over one phase sweep. Subtracting the value at $\rho = 0$ leaves
the *fluctuating* lift, without the base flow's own contribution.

In [ ]:
l_free, L0 = NSE.lift_functional(case)
L_coeffs, mset_L = NSE.lift_polynomial(W, l_free)
L = DensePolynomial(L_coeffs, mset_L)

θ = range(0, 2π; length = 257)[1:256]

function max_lift(P, ρ, η)
    base = evaluate(P, [0.0im, 0.0im, complex(η)])
    maximum(abs(real(evaluate(P, [ρ * cis(t), ρ * cis(-t), complex(η)]) - base)) for t in θ)
end

max_lift(L, branch.amplitude[end], branch.parameter[end]) # peak lift at the far end

## 7. Convergence with expansion order

The cohomological solve is **graded**: a degree-$N$ coefficient never depends on degrees above
$N$. Truncating the order-9 coefficients therefore gives the order-$N$ ROM exactly, and the
order study costs nothing beyond this one run. `restrict_ReducedDynamics_to_degree` truncates
`R`; the lift is truncated the same way, on the projected polynomial rather than on `W`, which
at order 9 is a ~193 MB array.

Where the curves lie on top of one another the truncation has converged. Where they separate,
the reduced state has left the region where the expansion converges, and a branch that folds
back does so as a truncation artifact rather than as physics.

MORFE draws nothing itself. `normal_form_branch` returns vectors, and the plotting below is an
explicit Makie call on them.

In [ ]:
using CairoMakie

D = case.fom.reference_length      # 0.1 m, the reference length in ν = D/Re
strouhal(Ω) = Ω * D / (2π * NSE.U_MEAN)

curves = map(3:2:order) do N
    b = normal_form_branch(restrict_ReducedDynamics_to_degree(R, N); sweep...)
    P = MORFE.Polynomials.restrict_polynomial_to_degree(L, N)
    (; N, b.amplitude, b.frequency, eta = b.parameter, Re = to_Re.(b.parameter),
        St = strouhal.(b.frequency), lift = max_lift.(Ref(P), b.amplitude, b.parameter))
end

fig = Figure(size = (900, 360))
ax_lift = Axis(fig[1, 1]; xlabel = "Re", ylabel = "max |lift|")
ax_st = Axis(fig[1, 2]; xlabel = "Re", ylabel = "Strouhal number")
for c in curves
    lines!(ax_lift, c.Re, c.lift; label = "order $(c.N)")
    lines!(ax_st, c.Re, c.St; label = "order $(c.N)")
end
axislegend(ax_lift; position = :lt)
fig

## Optional: save the ROM and the branch

The common saver writes `W`, `R`, their coefficient table and a summary to the example's
`results` directory; `validate.jl` reads the coefficient table it writes. `branch.csv` holds
every curve drawn above, and the order-9 file is what the
[Kármán tutorial page](https://morfeproject.github.io/MORFE.jl/tutorials/karman.html) plots.

The rows stay in amplitude order. Orders 5 and 9 fold, and a curve sorted by Re instead would
zig-zag across its own fold.

In [ ]:
MORFE.save_rom(joinpath(@__DIR__, "results"), W, R;
    external_system = meta.external_system)

open(joinpath(@__DIR__, "results", "data", "branch.csv"), "w") do io
    println(io, "order,eta,Re,rho,omega,St,max_abs_lift")
    for c in curves, i in eachindex(c.Re)
        println(io, join((c.N, c.eta[i], c.Re[i], c.amplitude[i], c.frequency[i],
            c.St[i], c.lift[i]), ","))
    end
end